# 02 — Synthetic Training Data Generation

Uses the **existing IPCC Eq. 2.25 engine** as a process-based model surrogate (analogous to Liu et al. using `ecosys` for 14M samples). Monte Carlo sampling of the IPCC parameter space generates 12,000 training samples with:
- Daily weather features (365 days × 3 variables)
- NDVI as GPP proxy (365 days)
- Soil properties (4 variables, static per farm)
- Target: `delta_soc_annual` from IPCC engine (t C/ha/yr)

In [ ]:
import sys, os
import numpy as np
import json
from pathlib import Path

# Add backend to path
backend_path = os.path.join(os.path.dirname(os.getcwd()), 'backend')
sys.path.insert(0, backend_path)

from app.models.farm import FarmInput
from app.core.mrv_engine import calculate_carbon
from app.data_loader import load_soc_factors

soc_data = load_soc_factors()
np.random.seed(42)

print('IPCC engine loaded. Generating synthetic training data...')

In [ ]:
# ── Sampling distributions ────────────────────────────────────────────
CROP_TYPES = ['tea_organic', 'tea_conventional', 'rubber_agroforestry',
              'paddy_rice', 'spice_cinnamon', 'coconut_organic', 'forest_regen']

CROP_PRACTICES = {
    'tea_organic':         'organic_conversion',
    'tea_conventional':    'conventional_management',
    'rubber_agroforestry': 'agroforestry_adoption',
    'paddy_rice':          'improved_water_management',
    'spice_cinnamon':      'reduced_till',
    'coconut_organic':     'organic_conversion',
    'forest_regen':        'forest_regen',
}

DISTRICTS = list(soc_data['district_climate_zone'].keys())
SOIL_TYPES = ['HAC', 'LAC', 'Sandy']
SOIL_WEIGHTS = [0.5, 0.3, 0.2]

# Climate zone parameters for synthetic weather
ZONE_WEATHER = {
    'tropical_montane': {'t_mean': 18.0, 't_amp': 3.0, 'precip_annual_mm': 2500, 'rad_base': 220},
    'tropical_wet':     {'t_mean': 27.0, 't_amp': 2.0, 'precip_annual_mm': 3500, 'rad_base': 250},
    'tropical_moist':   {'t_mean': 28.0, 't_amp': 3.0, 'precip_annual_mm': 2000, 'rad_base': 280},
    'tropical_dry':     {'t_mean': 30.0, 't_amp': 4.0, 'precip_annual_mm': 1200, 'rad_base': 310},
}

# NDVI seasonal patterns by crop
NDVI_PATTERNS = {
    'tea_organic':         {'base': 0.72, 'amp': 0.05, 'phase': 0},
    'tea_conventional':    {'base': 0.60, 'amp': 0.06, 'phase': 0},
    'rubber_agroforestry': {'base': 0.80, 'amp': 0.04, 'phase': 30},
    'paddy_rice':          {'base': 0.45, 'amp': 0.25, 'phase': 60},  # strong seasonal
    'spice_cinnamon':      {'base': 0.68, 'amp': 0.05, 'phase': 0},
    'coconut_organic':     {'base': 0.70, 'amp': 0.04, 'phase': 15},
    'forest_regen':        {'base': 0.85, 'amp': 0.03, 'phase': 0},
}

# Soil property defaults by climate zone (from OpenLandMap typical values)
ZONE_SOIL = {
    'tropical_montane': {'soc_g_kg': 25.0, 'bd': 1.1, 'ph': 5.2, 'clay': 35.0},
    'tropical_wet':     {'soc_g_kg': 18.0, 'bd': 1.2, 'ph': 5.5, 'clay': 40.0},
    'tropical_moist':   {'soc_g_kg': 12.0, 'bd': 1.3, 'ph': 6.0, 'clay': 30.0},
    'tropical_dry':     {'soc_g_kg': 6.0,  'bd': 1.4, 'ph': 6.8, 'clay': 20.0},
}

N_SAMPLES = 12000
print(f'Will generate {N_SAMPLES} samples across {len(CROP_TYPES)} crops, {len(DISTRICTS)} districts')

In [ ]:
# ── Generate samples ──────────────────────────────────────────────────
CROP_TO_ID = {c: i for i, c in enumerate(CROP_TYPES)}

features_all = []    # (N, 365, 8)
crop_ids_all = []    # (N,)
delta_soc_all = []   # (N,) — target from IPCC engine
meta_all = []        # metadata for analysis

errors = 0
for i in range(N_SAMPLES):
    # Random farm parameters
    crop = np.random.choice(CROP_TYPES)
    district = np.random.choice(DISTRICTS)
    soil = np.random.choice(SOIL_TYPES, p=SOIL_WEIGHTS)
    area = np.random.uniform(0.5, 10.0)
    fert = np.random.lognormal(3.0, 1.0) if crop not in ('tea_organic', 'coconut_organic', 'forest_regen') else 0.0
    fert = min(fert, 300.0)
    fuel = np.random.uniform(0, 150.0)
    
    # Run IPCC engine
    try:
        farm = FarmInput(
            land_area_ha=area,
            crop_type=crop,
            practice_change=CROP_PRACTICES[crop],
            district=district,
            soil_type=soil,
            fertiliser_kg_ha_yr=fert,
            fuel_litres_yr=fuel,
        )
        result = calculate_carbon(farm)
        delta_soc = result.delta_soc_annual  # t C/ha/yr
    except Exception:
        errors += 1
        continue
    
    # Generate synthetic daily weather (365 days)
    zone = soc_data['district_climate_zone'].get(district, 'tropical_moist')
    zw = ZONE_WEATHER[zone]
    days = np.arange(365)
    
    # Temperature: seasonal sinusoid + noise
    temp = zw['t_mean'] + zw['t_amp'] * np.sin(2 * np.pi * days / 365) + np.random.normal(0, 1.5, 365)
    
    # Precipitation: gamma distribution with seasonal modulation
    seasonal_mod = 1.0 + 0.5 * np.sin(2 * np.pi * (days - 120) / 365)  # peak at day 120
    daily_mean = zw['precip_annual_mm'] / 365
    precip = np.random.gamma(0.3, daily_mean / 0.3 * seasonal_mod)
    precip = np.clip(precip, 0, 100)
    
    # Radiation: anti-correlated with precipitation
    rad = zw['rad_base'] - 0.3 * precip + np.random.normal(0, 20, 365)
    rad = np.clip(rad, 50, 400)
    
    # NDVI: seasonal pattern per crop
    ndvi_p = NDVI_PATTERNS[crop]
    ndvi = ndvi_p['base'] + ndvi_p['amp'] * np.sin(2 * np.pi * (days - ndvi_p['phase']) / 365)
    ndvi += np.random.normal(0, 0.02, 365)
    ndvi = np.clip(ndvi, 0.05, 0.95)
    
    # Soil properties (static, repeated for each day)
    zs = ZONE_SOIL[zone]
    soc_val = zs['soc_g_kg'] * (1 + np.random.normal(0, 0.1))
    bd_val = zs['bd'] * (1 + np.random.normal(0, 0.05))
    ph_val = zs['ph'] * (1 + np.random.normal(0, 0.05))
    clay_val = zs['clay'] * (1 + np.random.normal(0, 0.1))
    
    # Stack into (365, 8) feature matrix
    daily_features = np.column_stack([
        temp,                                   # 0: temperature_c
        precip,                                 # 1: precipitation_mm
        rad,                                    # 2: radiation_wm2
        ndvi,                                   # 3: ndvi (GPP proxy)
        np.full(365, soc_val),                  # 4: soc_g_kg
        np.full(365, bd_val),                   # 5: bulk_density
        np.full(365, ph_val),                   # 6: ph
        np.full(365, clay_val),                 # 7: clay_pct
    ])  # shape: (365, 8)
    
    features_all.append(daily_features)
    crop_ids_all.append(CROP_TO_ID[crop])
    delta_soc_all.append(delta_soc)
    meta_all.append({'crop': crop, 'district': district, 'zone': zone, 'area': area})
    
    if (i + 1) % 3000 == 0:
        print(f'  Generated {i+1}/{N_SAMPLES} samples...')

features = np.array(features_all, dtype=np.float32)  # (N, 365, 8)
crop_ids = np.array(crop_ids_all, dtype=np.int64)    # (N,)
delta_soc = np.array(delta_soc_all, dtype=np.float32) # (N,)

print(f'\nGenerated {len(features)} samples (errors: {errors})')
print(f'Features shape: {features.shape}')
print(f'delta_SOC range: [{delta_soc.min():.4f}, {delta_soc.max():.4f}] t C/ha/yr')
print(f'delta_SOC mean: {delta_soc.mean():.4f} t C/ha/yr')

In [ ]:
# ── Generate synthetic daily NEE targets ──────────────────────────────
# Distribute annual delta_SOC across the year with seasonal weighting
# NEE convention: negative = sequestration (ecosystem absorbs C)
# Mass balance: ΔSOC = -NEE - Yield  →  NEE = -(ΔSOC + Yield)

daily_nee = np.zeros((len(features), 365), dtype=np.float32)
for idx in range(len(features)):
    # Growing season weight: higher sequestration when NDVI is high
    ndvi_series = features[idx, :, 3]  # NDVI column
    weight = ndvi_series / ndvi_series.sum()  # normalize to sum=1
    
    # annual NEE ≈ -(delta_soc + yield_proxy)
    # For training, we set yield_proxy = 0 so NEE = -delta_soc
    annual_nee = -delta_soc[idx]  # negative = sequestration
    daily_nee[idx] = annual_nee * weight  # distribute across year

print(f'Daily NEE shape: {daily_nee.shape}')
print(f'Sample annual sum: {daily_nee[0].sum():.4f} (should be ≈ {-delta_soc[0]:.4f})')

In [ ]:
# ── Z-normalize features ──────────────────────────────────────────────
# Save mean/std for inference-time normalization
feat_mean = features.reshape(-1, 8).mean(axis=0)
feat_std = features.reshape(-1, 8).std(axis=0)
feat_std[feat_std < 1e-6] = 1.0  # prevent division by zero

features_norm = (features - feat_mean) / feat_std

print('Feature normalization stats:')
names = ['temp', 'precip', 'rad', 'ndvi', 'soc', 'bd', 'ph', 'clay']
for i, n in enumerate(names):
    print(f'  {n:8s}: mean={feat_mean[i]:8.3f}  std={feat_std[i]:8.3f}')

In [ ]:
# ── Train/validation split ────────────────────────────────────────────
N = len(features)
n_train = int(0.85 * N)
indices = np.random.permutation(N)
train_idx = indices[:n_train]
val_idx = indices[n_train:]

print(f'Train: {len(train_idx)} samples')
print(f'Val:   {len(val_idx)} samples')

# ── Save everything ───────────────────────────────────────────────────
np.savez_compressed('data/synthetic_train.npz',
    features=features_norm.astype(np.float32),
    crop_ids=crop_ids,
    delta_soc=delta_soc,
    daily_nee=daily_nee,
    train_idx=train_idx,
    val_idx=val_idx,
    feat_mean=feat_mean,
    feat_std=feat_std,
)

# Save crop-to-id mapping + normalization stats for backend
config = {
    'crop_to_id': CROP_TO_ID,
    'feat_mean': feat_mean.tolist(),
    'feat_std': feat_std.tolist(),
    'feature_names': names,
    'n_train': len(train_idx),
    'n_val': len(val_idx),
}
with open('data/training_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print(f'\nSaved: data/synthetic_train.npz ({os.path.getsize("data/synthetic_train.npz")/1e6:.1f} MB)')
print(f'Saved: data/training_config.json')

In [ ]:
# ── Quick visualization ───────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# delta_SOC distribution
axes[0].hist(delta_soc, bins=50, color='#2ca02c', alpha=0.8)
axes[0].set_xlabel('delta_SOC (t C/ha/yr)')
axes[0].set_ylabel('Count')
axes[0].set_title('IPCC-derived delta_SOC distribution')

# delta_SOC by crop
import pandas as pd
df_meta = pd.DataFrame(meta_all)
df_meta['delta_soc'] = delta_soc[:len(df_meta)]
df_meta.boxplot(column='delta_soc', by='crop', ax=axes[1], rot=45)
axes[1].set_title('delta_SOC by Crop Type')
axes[1].set_xlabel('')

# Sample daily features
idx = 0
axes[2].plot(features[idx, :, 0], label='Temp (C)', alpha=0.7)
axes[2].plot(features[idx, :, 3] * 30, label='NDVI x 30', alpha=0.7)
axes[2].plot(features[idx, :, 1] / 5, label='Precip / 5', alpha=0.5)
axes[2].set_xlabel('Day of Year')
axes[2].set_title(f'Sample daily features (crop={meta_all[idx]["crop"]})')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig('data/synthetic_data_overview.png', dpi=150)
plt.show()